## 1. 인트로
### 1-1. RNN의 한계와 어텐션의 등장
#### [Seq2Seq 복습]
- 입력 시퀀스를 받아 다른 시퀀스로 변환하는 모델 구조
- 장점
    - 입력과 출력의 길이가 달라도 사용 가능 -> ex. 영어 3개 단어 -> 한국어 5개 단어
    - 번역, 요약, 음성 인식 등 다양한 변환 작업에 사용 가능
- Encoder
    - LSTM이 문장을 한 단어씩 순차적으로 읽음
    - 다 읽은 후 문장 전체 의미를 요약해서 하나의 벡터(Context Vector)로 압축
- Context Vector
    - 인코더가 만든 문장 요약본
    - 인코더와 디코더를 잇는 유일한 통로
- Decoder
    - Context Vector를 받아 번역문을 한 단어씩 만듦
    - `<sos>` 넣어서 "나는" 출력 -> "나는"을 다시 입력으로 넣어서 "수학을" 출력 -> 반복 후 `<eos>`로 멈춤
- 디코더가 참고할 수 있는 건 Context Vector 하나뿐

#### [Attention 등장 배경 : Seq2Seq의 병목 현상]
- Context Vector 크기가 고정 -> 5단어든 50단어든 같은 크기 벡터에 다 담아야 함 -> 문장 길어질수록 정보 손실
- "나는"을 만들 때도, "수학을"을 만들 때도 똑같은 요약본 참고 -> 출력 단어마다 필요한 정보를 다르게 반영 X
- Context Vector = 인코더의 마지막 hidden state -> 마지막 단어 정보는 진하고, 앞쪽 단어 정보는 희미해짐
- 해결 -> Attention
    - 디코더가 단어 생성할 때마다 입력 문장 전체를 다시 보고, 관련 있는 부분에 집중
    - 출력 단어마다 새로운 Context Vector 생성

#### [Attention 정의]
- 현재 시점에서 가장 중요한 부분에 더 높은 가중치를 부여하는 기술
- 출력 단어 생성할 때 입력 문장의 어떤 단어에 더 집중할지 정도를 결정
- 기존 Seq2Seq와 달리 마지막 hidden state 하나가 아니라 모든 hidden state를 남겨둠
- 디코더의 현재 상태를 인코더의 각 hidden state와 비교해서 관련도 점수 계산 -> Softmax -> 비율만큼 섞어서 Context Vector 생성
- 각 단어와의 관련도 점수는 서로 기다릴 필요 없이 동시에 계산 가능 -> 병렬화 가능성
- 단, 인코더와 디코더는 여전히 LSTM이라 모델 전체 병렬화는 X -> Attention은 RNN의 보조 역할
- LSTM 걷어내고 어텐션만 남긴 모델 = Transformer

#### [용어 정리]
- Hidden state : RNN/LSTM이 단어 읽을 때마다 갱신하는, 그 시점까지의 문맥 담은 벡터
- Context Vector : 디코더가 출력 단어 만들 때 참고하는 원문 정보 벡터
- Attention Score : 디코더의 현재 상태와 인코더 각 단어 사이의 관련도 점수
- Softmax : 여러 점수를 0~1 사이, 합계 1인 비율로 변환하는 함수
- Attention Weight : Softmax 거친 비율, 각 단어를 얼마나 참고할지

#### [Attention 과정]
1. 입력을 벡터 형태로 변환
2. 단어별 중요도 계산
    - 관련도 점수 : 현재 작업과 각 단어가 얼마나 관련 있고 유사한지 점수 매김
    - Softmax : 관련도 점수를 합이 1인 비율로 변환 -> Attention Weight
    - ex. student 관련도 점수 3.8 -> Softmax 거쳐서 80% -> student에 80% 집중
3. 중요도에 따른 입력 정보 조합
    - 어텐션 가중치만큼 각 단어의 hidden state를 섞어서 Context Vector 생성
    - 비율 높은 단어 정보 위주로 반영

#### [Attention 장점]
- 시간 경과에 대한 유연성 (vs. RNN)
    - RNN은 멀리 떨어진 단어일수록 정보가 흐려짐 (장기 의존성 문제)
    - Attention은 모든 단어를 직접 비교 -> 거리 상관없이 관계 파악 가능
- 공간에 대한 유연성 (vs. CNN)
    - CNN은 고정된 크기의 필터로 주변 정보만 봄 -> 멀리 있는 정보 연결하려면 층 여러 번 쌓아야 함
    - Attention은 모든 위치를 직접 연결 -> 한 번에 전역적인 관계 학습
- 병렬화 (vs. RNN)
    - RNN은 앞 단계 끝나야 다음 단계 계산 -> 학습 느림
    - Attention의 관련도 계산은 단어 쌍마다 독립적 -> 동시에 계산 가능

#### [정리]
- RNN -> 장기 의존성 문제 + 병렬화 불가
- LSTM -> 장기 의존성은 완화, but 순차 구조는 그대로
- Seq2Seq + Attention -> 병목 현상은 해결, but 여전히 RNN 위에서 작동하는 보조 역할


### 1-2. Attention is All You Need
#### [논문 소개]
- Transformer 구조를 처음 발표한 논문 (Self-Attention 방식)
- 2017년 NIPS, Google Brain·Google Research 연구진 8명 발표
- "Attention만 있으면 된다" -> 오늘날 LLM의 기반이 된 논문
- 배경 : RNN, LSTM, GRU는 n번째 단어 계산하려면 n-1번째 결과가 먼저 나와야 함 -> 병렬 처리 불가
- RNN과 CNN을 완전히 제거하고 Self-Attention으로 입력과 출력의 관계를 계산하는 Transformer 제안
- 가장 중요한 발견 : Recurrence(순환)와 Convolution(합성곱)은 필수가 아니다
    - Attention만으로 더 좋은 성능
    - 순차 구조 없앤 덕분에 병렬화로 빠른 학습 가능
- 파급효과
    - LLM 탄생 : 디코더 쌓은 GPT, 인코더 쌓은 BERT
    - NLP 평정 : 번역, 요약, 질의응답 등 대부분의 표준 구조
    - 타 분야 확장 : 이미지, 음성 등

#### [Transformer란?]
- 단어를 순서대로 처리하지 않고 Attention만으로 문장 전체 의미와 구조를 파악해서 한 번에 처리하는 인코더-디코더 모델

    <img src="https://upload.wikimedia.org/wikipedia/commons/3/34/Transformer,_full_architecture.png" width="400" style="background-color:white; padding:10px;">

- 왼쪽(인코더) : 입력 문장 이해
- 오른쪽(디코더) : 출력 문장 생성
- 층 하나를 N개 쌓아서 인코더와 디코더 구성
- 기존 모델과의 차이 = 인코더와 디코더 내부를 무엇으로 채웠는가

| 구분 | Seq2Seq + Attention | Transformer |
|:---:|:---:|:---:|
| 인코더, 디코더 내부 | LSTM (순차 처리) | Self-Attention |
| 어텐션의 역할 | RNN을 돕는 보조 역할 | 모델 그 자체 |
| 어텐션 사용 위치 | 디코더 -> 인코더 1곳 | 인코더 내부, 디코더 내부, 인코더-디코더 사이 3곳 |
| 단어 순서 정보 | RNN이 순서대로 읽으며 반영 | Positional Encoding으로 따로 추가 |
| 병렬 처리 | 불가 | 가능 |

#### [Transformer 핵심 기술]
- Self-Attention : 문장 안의 단어들이 서로 참고해서 한 번에 문맥 파악
- Multi-Head Attention : Attention을 여러 갈래로 나눠서 다양한 관점에서 관계 파악
- Positional Encoding : 각 단어 위치를 벡터로 만들어 단어 벡터에 더함 -> 사라진 순서 정보 보완


## 2. Self-Attention
### 2-1. Query, Key, Value
#### [RNN 복습]
- 이전 단계에서 처리한 정보를 현재 단계 입력에 함께 활용하는 신경망
- $X_{t-1}$ 시점의 정보 $h_{t-1}$이 다음 시점 $h_t$로 전달 -> 과거 정보 계속 전달 (= 기억)
- 가중치 공유 : 모든 타임 스텝에서 동일한 가중치 반복 사용 -> Parameter 효율성, 학습 일반화 가능
- 장기 의존성 문제 : 데이터 길어질수록 초반 정보가 후반부에 잘 반영 X
- Vanishing gradient
    - BPTT(Backpropagation Through Time)로 학습 -> 문장 길수록 곱해지는 가중치 횟수 증가
    - 계산 값 발산 -> Exploding gradient
    - 계산 값 0 -> Vanishing gradient -> 초반부 데이터가 후반부에 반영 X

#### [LSTM 복습]
- "중요한 것은 오래 기억하고, 필요 없는 것은 잊어버리자"
- Cell state : 가중치 곱 거치지 않고 직선적으로 전달되는 정보 -> 중요한 정보 적는 메모장
- Gate
    - Input gate : 새로운 정보 중 중요한 것만 Cell state로
    - Forget gate : 이전 정보 중 필요 없는 정보 제거
    - Output gate : 이번 output에 필요한 정보만 뽑아냄
- 장점 : 오래된 정보도 기억 가능, gradient 유지, 선택적으로 저장/삭제
- 단점 : 게이트 3개라 계산 복잡하고 느림, 긴 문장에서는 여전히 Vanishing gradient 발생 (초반부 데이터가 forget gate를 더 많이 통과해야 해서)
- LSTM은 기억 전달 방식은 개선했지만 순서대로 처리하는 구조 자체는 못 바꿈
- Transformer는 순서대로 전달하는 대신 모든 단어를 직접 비교 -> 이때 쓰는 재료 = Q, K, V

#### [Q, K, V]
- Attention = Query를 Key들과 비교해서 관련도 구하고, 그 비율만큼 Value를 섞어 결과 만드는 계산
- Query
    - 현재 단어가 다른 단어들에게 "지금 나한테 필요한 정보가 뭐지?"라고 묻는 벡터
    - ex. 도서관에서 "'학생'에 대해 알고 싶어"라는 궁금증
- Key
    - 각 단어가 "나는 이런 정보를 가지고 있어"라고 보이는 벡터, 검색 대상의 '이름표'
    - Query와 비교해서 관련도 점수 결정
    - ex. '학교 생활', '교육학 개론', '요리 레시피', '주식 투자' 제목표 -> 60%, 35%, 3%, 2%
- Value
    - 각 단어가 실제로 전달하는 정보 벡터
    - 관련도 비율만큼 모든 Value 섞어서 최종 결과
    - ex. 학교 생활 내용 60% + 교육학 개론 35% + 요리 레시피 3% + 주식 투자 2% -> 최종 답변
- Key와 Value는 한 단어에서 함께 만들어진 한 쌍
    - Key = 찾을 때 쓰는 기준, Value = 찾았을 때 가져오는 내용
    - 나누는 이유 : 같은 단어라도 "관련 있는지 판단할 때 필요한 정보"와 "실제로 전달할 정보"가 다를 수 있기 때문


### 2-2. Cross-Attention vs. Self-Attention
- 공통 흐름
    1. 입력을 학습된 변환 $W_Q, W_K, W_V$에 통과시켜 Q, K, V 생성
    2. Query x Key -> 모든 단어 쌍의 관련도 표
    3. Softmax -> 각 줄 합이 1인 비율로 변환
    4. 비율 x Value -> 새로운 hidden states
- 차이점
    - Self-Attention : Q, K, V 모두 같은 입력에서 생성
    - Cross-Attention : Q는 디코더, K와 V는 인코더 출력에서 생성

#### [Cross-Attention (기존 Seq2Seq의 Attention)]
- Query와 Key-Value 쌍이 서로 다른 시퀀스에서 나옴
- 디코더가 인코더에게 "지금 어디를 봐야 하지?"라고 묻는 구조
- ex. "The cat is sleeping" -> "고양이가 ____" 다음 단어 만드는 순간
    1. Query : 디코더가 지금까지 만든 "고양이가"를 바탕으로 현재 상태를 Query로
    2. Key, Value : 인코더가 원문 각 단어(The, cat, is, sleeping)에 대해 만든 벡터
    3. 유사도 계산 : Query와 각 Key 내적($Q \cdot K_i$) -> Softmax -> The 2%, cat 13%, is 10%, sleeping 75%
    4. Weighted Sum : 비율만큼 Value 섞어서 Context Vector 생성 -> "자고" 출력
- 오픈북 시험처럼 답안지 쓰면서 교과서 어디를 볼지 계속 정하는 과정
- 한계 : 인코더, 디코더가 RNN 위에서 작동 -> 원문 내부 관계는 간접적으로 파악, 병렬 처리 불가

#### [Self-Attention]
- Query와 Key-Value 쌍이 같은 시퀀스에서 나옴
- 문장이 스스로에게 "우리끼리 누가 누구와 관련이 있지?"라고 묻는 구조
- RNN 없이도 문장 내부 관계와 문맥 직접 파악 가능

    <img src="https://upload.wikimedia.org/wikipedia/commons/7/74/Encoder_self-attention,_detailed_diagram.png" width="450" style="background-color:white; padding:10px;">

- ex. "가을 은행 냄새 지독해"에서 '은행'의 의미 파악
    1. 벡터 임베딩 : 각 단어를 벡터로 바꾸고 같은 벡터에서 Q, K, V 생성
    2. 내적 및 관련도 점수 : '은행'의 Query와 모든 단어의 Key 비교 (가로줄 = 질문하는 단어, 세로줄 = 참고 대상)
    3. 중요도 배분 : 가을 30%, 은행 20%, 냄새 45%, 지독해 5%
    4. 새로운 벡터 생성 : '은행'이 "돈? 열매?"에서 "가을에 냄새 나는 열매"라는 의미를 담은 새 벡터가 됨
- 모든 단어가 동시에 이 과정 거침 -> 병렬 처리 가능

#### [비교]
| 구분 | Cross-Attention | Self-Attention |
|:---:|:---:|:---:|
| point | 번역 중인 문장이 원문을 참고 | 문장이 스스로 내부 관계 파악 |
| 참조 대상 | 서로 다른 시퀀스 (디코더 -> 인코더) | 같은 시퀀스 (문장 내부) |
| 주요 목적 | 고정 Context Vector의 병목 해결 | 문맥이 반영된 단어 표현 생성 |
| 결과 벡터 | Context Vector | 문맥이 반영된 새 표현 |
| 병렬 처리 | RNN 기반일 때 불가능 | 가능 |

- Transformer는 두 방식 모두 사용
    - 인코더 : Self-Attention -> 원문 단어끼리 관계 파악
    - 디코더 : Self-Attention -> 번역 중인 문장끼리 관계 파악 (아직 안 만든 뒤쪽 단어는 못 보게 가림)
    - 인코더-디코더 사이 : Cross-Attention -> 번역하면서 원문 참고
- Cross는 "남에게 묻기", Self는 "나에게 묻기"


### 2-3. Scaled Dot-Product Attention
#### [Dot-Product Attention]
- Query와 Key 벡터를 내적해서 관련도 점수 구하는 방식
- 내적 : 두 벡터의 같은 위치 성분끼리 곱해서 모두 더한 값 -> 방향 비슷할수록 큰 값 -> "얼마나 닮았나"

$$ Q \cdot K = \sum_{i=1}^{d_k} Q_i K_i $$

- 선대 개념
    - 행렬 : 숫자를 행과 열로 배열, Attention에서는 한 행 = 한 단어의 벡터 -> ex. Q (6×3) : 단어 6개가 각각 3차원 벡터
    - 전치(Transpose) : 행과 열을 바꿈 -> K (6×3)를 $K^T$ (3×6)로
    - 행렬 곱 조건 : 앞 행렬의 열 개수 = 뒤 행렬의 행 개수 -> Q (6×3) × $K^T$ (3×6) = (6×6), 그래서 K를 전치함
    - 행렬 곱 = 내적의 모음 -> $QK^T$의 i행 j열 = i번째 단어 Query와 j번째 단어 Key의 내적
- 과정
    1. Attention Score 계산 : Q (6×3) × $K^T$ (3×6) = (6×6) 관련도 점수표 -> 행렬 곱 한 번으로 36개 점수 동시 계산 (병렬 처리가 실제로 구현되는 지점)
    2. Softmax 적용 : 각 가로줄을 합이 1인 비율로 -> Attention Weight
    3. Attention Weight (6×6) × Value -> 각 단어가 다른 단어 정보를 비율만큼 섞은 새 벡터 6개
- 문제점 : Attention Score의 극단화
    - 벡터 차원 커질수록 내적 값이 지나치게 커짐
    - Softmax에 들어가면 특정 단어에 확률이 과도하게 몰림
    - 기울기 0에 가까워져서 Vanishing Gradient 문제

#### [Scaled Dot-Product Attention]
- "Attention Is All You Need" 저자들이 제안, Transformer의 Attention 계산 공식 그 자체
- 내적과 Softmax 사이에 스케일링 단계 추가
    1. Query와 모든 Key 내적 -> Attention Score
    2. Attention Score를 $\sqrt{d_k}$로 나눔 (스케일링)
    3. Softmax -> Attention Weight
    4. Attention Weight × Value -> 최종 결과
- $\sqrt{d_k}$로 나누면 분산이 다시 1 근처로 -> 차원 크기 상관없이 점수가 안정적인 범위에 머묾
- ex. 스케일링 전엔 sheepdog 하나에 82.2% 몰림 -> 스케일링 후엔 sheepdog 64.5%, shetland 19.9%, affectionate 15.7%로 나눠짐 -> 주변 정보 고르게 반영

$$ Attention(Q, K, V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

- $QK^T$ : 모든 Query와 Key의 내적 -> 관련도 점수표
- $\sqrt{d_k}$ : 점수 크기 조정 (Scaling)
- $V$ : 비율만큼 Value 섞어서 결과 생성
- 결과 벡터
    - Cross-Attention : Context Vector -> 현재 생성 중인 단어에 필요한 원문 정보를 비율만큼 모은 벡터
    - Self-Attention : 문장 내 다른 단어들과의 관계가 반영되어 의미가 업데이트된 단어 벡터


## 3. Multi-Head Attention
### 3-1. 왜 Multi-Head Attention을 사용할까?
- 헤드(Head) : Attention 계산을 한 번 하는 독립된 단위
    - 자기만의 가중치 행렬 세트 $W^Q, W^K, W^V$로 Q, K, V 만들고 Attention 한 번 계산 -> 결과 벡터 하나
    - 헤드마다 가중치 행렬 숫자가 다름 -> 같은 문장 봐도 집중하는 단어가 달라짐

#### [Single-Head Attention]
- 헤드 1개, 512차원 벡터를 그대로 한 번 Attention -> 단어마다 가중치 분포 하나
- 방식
    1. 512차원 벡터에 $W^Q, W^K, W^V$ 한 세트만 곱해서 Q, K, V 생성
    2. Q, K로 점수 매기고 스케일링 후 Softmax -> 합이 1인 가중치 분포 하나
    3. 가중치로 V 섞어서 새 512차원 벡터 하나 출력
- 한계 : 집중력 100%를 혼자 나눠 써야 함
    - 한 단어 이해하려면 여러 종류의 관계가 동시에 필요한 경우가 많음
    - 100%를 여러 관계에 나눠 줄 수밖에 없음 -> 여러 정보가 섞여 흐릿해진 벡터
    - ex. "Tom gave Mary a book yesterday"에서 "gave" -> Tom 0.25, Mary 0.25, book 0.25, yesterday 0.23 -> 준 사람이 Tom인지 Mary인지 흐려짐

#### [Multi-Head Attention]
- Attention을 작게 쪼개서 여러 번 동시에 하는 방식
- 방식
    1. 512차원 벡터에 헤드마다 다른 $W^Q, W^K, W^V$ 곱해서 64차원 Q, K, V 8세트 생성
    2. 8개 헤드가 서로 간섭 없이 각자 Attention 계산 -> 가중치 분포 8개
    3. 8개의 64차원 결과를 이어 붙여(Concat) 다시 512차원
    4. $W^O$ 곱해서 8개 관점을 하나로 섞음
- 헤드마다 각자 100%를 씀
    - ex. 헤드1 -> Tom(Who?), 헤드2 -> Mary(To whom?), 헤드3 -> book(What?), 헤드4 -> yesterday(When?) 각각 0.90
    - 헤드마다 질문 하나에 집중하고 결과를 나란히 이어 붙임 -> 정보가 섞이지 않고 또렷하게 남음
- 비유
    - Single-Head = 기자 1명이 모든 질문을 혼자 취재 -> 다 담지만 깊이가 얕음
    - Multi-Head = 기자 8명이 각자 한 분야만 깊게 취재 -> 편집장($W^O$)이 8편을 엮어 한 기사로

### 3-2. Multi-Head Attention 작동 방식
- "I love you"에서 love가 통과하는 과정 (단어 512차원, 헤드 8개, 헤드당 64차원)
1. 분할(Projection) : 512 -> 64 × 8세트
    - love의 512차원 벡터에 헤드마다 다른 가중치 행렬 곱해서 64차원 Q, K, V 8세트
    - I, you도 똑같이 8세트씩
2. 병렬 어텐션 계산 : 8개 헤드가 동시에, 서로 간섭 없이 Scaled Dot-Product Attention 계산
    - 점수 매기기 : love의 $Q_1$을 I, love, you의 $K_1$과 각각 내적 -> 점수 3개
    - 크기 조절 : 헤드 차원의 제곱근($\sqrt{64} = 8$)으로 나눔
    - 가중치 변환 : Softmax -> ex. I 0.2, love 0.7, you 0.1
    - 섞기 : 가중치대로 $V_1$에 곱해서 더함 -> 헤드 1의 결과 (64차원)
    - 헤드 2~8도 동시에 독립적으로 계산 -> 64차원 결과 8개
3. 결합(Concat) : 64 × 8 -> 512, 8개 결과를 한 줄로 이어 붙임
4. 최종 투영(Final Projection) : 512 -> 512
    - 이어 붙인 벡터는 각 헤드 결과를 단순히 나열한 것
    - $W^O$ (512×512) 곱해서 8개 관점을 하나의 의미로 융합
- 기자 비유 : 취재 수첩 배분 -> 각자 분야 취재 -> 기사 8편 묶음 -> 편집장이 한 기사로 다듬기


## 4. Transformer 전체 아키텍처
### 4-1. 트랜스포머의 전처리 단계
- 텍스트 입력 -> 토큰화 -> 토큰 임베딩 -> Positional Encoding

#### [토큰화 (Tokenization)]
- 문장을 모델이 다루는 기본 단위(토큰)로 자르는 것
    - 보통 단어 하나 = 토큰 하나 -> ex. Give, me, an, apple
    - 문장 부호도 토큰
    - 길거나 드문 단어는 여러 조각 -> ex. didn't -> did + n't
- 레고 블록처럼 자주 쓰는 조각 준비해 두면 처음 보는 단어도 조립해서 표현 가능
- 논문 : 영->독 번역은 BPE 방식 약 37,000개, 영->불 번역은 word-piece 방식 32,000개 토큰 사전

#### [임베딩 (Embedding)]
- 각 토큰을 숫자(벡터)로 바꾸는 것
- 벡터의 길이 = 벡터의 차원 -> ex. 길이 1,024면 $d_{model}$ = 1,024
- 핵심 원리 : 뜻이 비슷한 단어는 가까운 좌표에 놓임
    - 처음엔 무작위, 학습하면서 의미에 맞는 자리를 찾아감
    - 단어마다 기본적인 의미가 담겨서 이후 Attention이 단어 사이 관계 계산 가능
    - ex. 딸기(5, 4), 체리(6, 6)는 '과일 동네' 이웃, 성(2, 1)은 멀리 '건물 동네'
    - 실제로는 512차원이라 색깔, 먹을 수 있는지, 크기 등 수백 가지 특징을 동시에 담음
- Word2Vec : 단어(Word)를 벡터(Vec)로 변환, 가장 일반적이고 강력한 임베딩 생성 방법 중 하나
    1. 신경망에 문장 입력하고 다음에 올 단어 예측하도록 훈련
    2. 각 층이 단어의 점점 더 깊은 속성 이해
    3. 마지막에서 두 번째 레이어(penultimate layer)의 숫자들을 단어의 임베딩으로 사용
    4. 이 레이어가 단어의 심층적인 속성을 포착 -> 좋은 임베딩
    - '딸기'와 '사과'는 비슷한 다음 단어를 예측 -> 추출되는 벡터도 비슷해짐

#### [Positional Encoding]
- 단어들의 순서 정보를 임베딩 벡터에 추가하는 단계
- 필요한 이유
    - Transformer는 모든 단어를 한꺼번에 봄 -> 그대로 두면 순서 모름
    - "개는 고양이를 쫓는다"와 "고양이는 개를 쫓는다"를 구분 못 함
    - RNN은 순서대로 읽어서 자연스럽게 알았지만, Transformer는 따로 알려줘야 함
- 방법
    - 각 위치마다 정해진 512차원 '위치 벡터'를 임베딩에 더함 (이어 붙이는 게 아니라 숫자끼리 더함)
    - 문장이 달라도 1번째 자리엔 항상 벡터 A, 2번째 자리엔 항상 벡터 B -> 규칙이 같아야 순서 학습 가능
    - 같은 단어라도 자리가 다르면 최종 벡터가 달라짐


### 4-2. 트랜스포머의 Encoder와 Decoder
#### [Encoder : 원문을 깊이 이해하는 분석팀]

<img src="https://upload.wikimedia.org/wikipedia/commons/9/92/Transformer,_one_encoder_block.png" width="200" style="background-color:white; padding:10px;">

- Encoder층 하나 = 두 부품
    1. Multi-Head Self-Attention : 원문 단어끼리 서로 보며 문맥 파악
    2. Feed-Forward Network : 단어 하나하나를 따로 더 깊이 가공
- RNN Encoder와 차이
    - 읽는 방식 : 하나씩 순서대로 vs. 한꺼번에
    - 정보 전달 : 이전 은닉 상태가 다음으로 vs. Self-Attention으로 모든 단어 직접 연결
    - 순서 정보 : 읽는 순서 자체에 들어 있음 vs. Positional Encoding으로 따로
    - 최종 출력 : 마지막 은닉 상태 하나 vs. 단어별 문맥 벡터 전부
- Feed-Forward Network(FFN)

$$ FFN(x) = \max(0, xW_1 + b_1)W_2 + b_2 $$

- 512차원 -> 2048차원으로 넓혔다가 다시 512차원으로 좁히는 2층 신경망, 사이에 ReLU
- 단어마다 따로, 같은 가중치로 적용 -> 이 단계에서는 단어끼리 정보 주고받지 X
- 비유
    - Self-Attention = 조별 토론, 학생(단어)들이 서로 의견 들으며 생각 업데이트
    - FFN = 토론 끝난 뒤 1:1 첨삭, 학생 한 명씩 같은 선생님에게 따로 가서 노트 다듬음
- Add & Norm : 모든 부품에 붙는 안전장치
    - Add = Residual Connection(잔차 연결) : 부품 통과한 결과에 원래 입력을 그대로 더함 (ResNet에서 나온 아이디어) -> 원본 보존, 변경 내용만 덧붙임
    - Norm = Layer Normalization(층 정규화) : 한 단어 벡터 안의 512개 숫자를 평균 0, 분산 1 근처로 -> 마이크 볼륨 맞추기, 학습 안정적이고 빠르게

#### [Decoder : 번역문을 한 단어씩 쓰는 작성팀]

<img src="https://upload.wikimedia.org/wikipedia/commons/5/55/Transformer,_one_decoder_block.png" width="230" style="background-color:white; padding:10px;">

- Decoder층 하나 = 세 부품
    1. Masked Multi-Head Self-Attention : 지금까지 쓴 번역 단어끼리 관계 파악 (미래 단어는 가림)
    2. Encoder-Decoder Attention (Cross-Attention) : 원문 분석 자료(Encoder 출력)에서 필요한 부분 찾기
    3. Feed-Forward Network : Encoder와 동일
- 세 부품 뒤에도 각각 Add & Norm, 6층 다 통과하면 Linear + Softmax가 다음 단어 확률 계산
- Masked Multi-Head Attention
    - Decoder의 Self-Attention에서 미래 시점 단어 정보를 가려서(Softmax 후 0) 자기보다 앞 단어만 보게 함
    - 학습 때는 정답 문장 전체를 한 번에 넣어 병렬로 학습 -> 마스크 없으면 답을 미리 보는 "컨닝"
    - ex. "나는 학교에 간다" -> "I go to school"에서 I go to 다음 단어 예측할 때 school을 미리 보면 컨닝

    <img src="https://upload.wikimedia.org/wikipedia/commons/7/72/Decoder_self-attention_with_causal_masking,_detailed_diagram.png" width="450" style="background-color:white; padding:10px;">

- Encoder-Decoder Attention (Cross-Attention)
    - 출력 문장(Decoder)과 입력 문장(Encoder)을 연결
    - Q = 직전 Decoder 서브레이어의 출력
    - K, V = Encoder 마지막 층의 출력 (6개 디코더 레이어가 모두 같은 Encoder 출력 사용)
    - ex. "school" 만들 차례 -> Query는 "I go to"까지의 문맥, Key/Value는 "나는 학교에 간다" 인코딩 벡터 -> "학교"에 높은 가중치


### 4-3. 전체 모델 및 데이터 흐름
- "I love you" 번역 흐름
- 입력 준비 : [의미 + 위치] 벡터로
    - 임베딩 : I, love, you를 각각 의미 벡터로 변환
    - Positional Encoding : 각 위치 벡터를 더함 -> "love"는 "2번째 자리의 love"
- 인코더 : 문맥적 의미 깊이 이해
    - 6개 레이어 통과하며 Multi-Head Self-Attention으로 "love"가 주어 "I", 목적어 "you"와의 관계 흡수, FFN으로 다듬음
    - 레이어 1개 안에서 : Multi-Head Self-Attention -> Add & Norm -> Feed-Forward (512 -> 2048 -> 512) -> Add & Norm
    - 출력 : 문맥적 의미 담은 벡터 -> Decoder의 K, V로 전달
- 디코더 : 한 단어씩 생성
    - 시작 토큰 -> Masked Self-Attention(지금까지 만든 단어만) -> Cross-Attention(Encoder 출력 참조) -> FFN, 6층 반복
    - Linear + Softmax로 어휘 전체 확률 구해서 가장 높은 단어 출력
    - Step 1 : `<start>`만 봄 -> I에 집중 -> "나는"
    - Step 2 : `<start>` 나는 -> 주어까지 썼다는 문맥 파악 -> you에 집중 -> "당신을"
    - Step 3 : `<start>` 나는 당신을 -> 주어와 목적어까지 썼다는 문맥 파악 -> love에 집중 -> "사랑해"
    - Step 4 : 문장 완성, 원문에서 더 옮길 내용 없음 -> `<end>` -> 종료
- 결과 : "나는 당신을 사랑해"
